In [1]:
import sys, os, random
sys.path.append(os.path.join(os.getcwd(), 'CONFOLD')) #add CONFOLD to path

import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
import pandas as pd

from foldrm import Classifier
from utils import split_data # Or your stratified version if you prefer
#from ModifiedClass import  extinction_birds2 # Our new function
from datasets import extinction_birds2, extinction_birds15

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [ ]:
random.seed(42)
random.seed(2)

In [3]:
model_template, data = extinction_birds2()

print(data.info())

data_dummies = pd.get_dummies(data)



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12613 entries, 0 to 12612
Data columns (total 38 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Primary.Lifestyle          12613 non-null  object 
 1   RLM                        12613 non-null  object 
 2   Nest_Type                  12613 non-null  object 
 3   Nest_SBS                   12613 non-null  object 
 4   Flightlessness             12613 non-null  object 
 5   Family                     12613 non-null  object 
 6   Foraging                   12613 non-null  object 
 7   MatingSystem               12613 non-null  object 
 8   NestPlacement              12613 non-null  object 
 9   Territoriality             12613 non-null  object 
 10  IslandDwelling             12613 non-null  object 
 11  Order1                     12613 non-null  object 
 12  Diet                       12613 non-null  object 
 13  Habitat_cat                12613 non-null  obj

In [4]:
#X = data.drop('Threat', axis=1).values.tolist()
X = data[model_template.attrs].drop('extinction_risk',axis=1)
X = X.convert_dtypes()
X = X.to_numpy()

y = data['extinction_risk'].to_numpy()

#for i in range(len(y)):
#    y[i]=str(y[i])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [5]:
#baseline_model = Classifier(attrs=model.attrs, numeric=model.numeric, label=model.label)
baseline_model = Classifier(attrs=model_template.attrs, numeric=model_template.numeric, label=model_template.label)
train_data = np.concatenate((np.array(X_train), np.array(y_train).reshape(-1, 1)), axis=1).tolist()
test_data = np.concatenate((np.array(X_test), np.array(y_test).reshape(-1, 1)), axis=1).tolist()

In [6]:
baseline_model.fit(train_data, ratio=0.5)
# Print the rules the model learned
print("--- Rules Learned by the Baseline Model ---")
baseline_model.print_asp(simple=True)




--- Rules Learned by the Baseline Model ---
extinction_risk(X,'lower_risk') :- range_size(X,N34), N34>54913.11287, not ab2(X), not ab9(X), not ab10(X), not ab14(X), not ab17(X), not ab19(X), not ab24(X). [confidence: 0.94924]
extinction_risk(X,'lower_risk') :- range_size(X,N34), N34<=54825.57, not ab27(X), not ab28(X), not ab34(X), not ab38(X), not ab41(X), not ab42(X), not ab43(X), not ab44(X), not ab46(X), not ab47(X), not ab49(X). [confidence: 0.8071]
extinction_risk(X,'higher_risk') :- range_size(X,N34), N34<=3161.83. [confidence: 0.79207]
extinction_risk(X,'higher_risk') :- not primary.lifestyle(X,'aerial'). [confidence: 0.9806]
extinction_risk(X,'higher_risk') :- primary.lifestyle(X,'aerial'). [confidence: 0.73529]
ab1(X) :- hand.wing.index(X,N23), N23>68.3.
ab2(X) :- wing.length(X,N20), N20>199.16826, tail.length(X,N24), N24>174.8, max.latitude(X,N27), N27<=-13.43, body_mass(X,N35), N35>1902.5, not ab1(X).
ab3(X) :- primary.lifestyle(X,'aquatic').
ab4(X) :- hb(X,N30), N30>5.0.
a

In [7]:
# Prepare the test data (features and true labels)
X_test = [d[:-1] for d in test_data]
Y_test = [d[-1] for d in test_data]

# Get predictions (these will be tuples of (label, confidence))
predictions_tuples = baseline_model.predict(X_test)
predicted_labels = [p[0] for p in predictions_tuples]

# Calculate accuracy
correct_predictions = 0
for i in range(len(Y_test)):
    if predicted_labels[i] == Y_test[i]:
            correct_predictions += 1

accuracy = correct_predictions / len(Y_test)

print("--- Baseline Model Evaluation ---")
#print(f"True Labels:    {Y_test}")
#print(f"Predicted Labels: {predicted_labels}")
print(f"Accuracy: {accuracy * 100:.2f}%")

# Instantiate a new classifier for our expert-guided model
expert_model = Classifier(attrs=model_template.attrs, numeric=model_template.numeric, label=model_template.label)

# Define our expert rules as strings
# Note: the symbols '==' and '<=' must also be in single quotes for the parser.
#rule1 = "with confidence 0.90 class = 'Higher_risk' if 'Range_size' '<=' '5'"
#Note additional rules could be added like this:
#rule2 = "with confidence 0.70 class = '1' if 'Clutch_Max' '==' '1'"

# Add the manual rules to the model
#expert_model.add_manual_rule(rule1, model_template.attrs, model_template.numeric, ['Lower_risk', 'Higher_risk'], instructions=False)
# Note: here is code to add an additional rule:
#expert_model.add_manual_rule(rule2, model_template.attrs, model_template.numeric, ['0', '1'], instructions=False)

print("--- Manual Rules Added to the Model (Before Training) ---")
# The internal representation is a bit complex, but we can see our rules are in there.
#for rule in expert_model.rules:
#    print(rule)



--- Baseline Model Evaluation ---
Accuracy: 89.18%
--- Manual Rules Added to the Model (Before Training) ---


In [8]:
# Now, fit the model on the training data.
# The algorithm will work around the rules we provided.
expert_model.fit(train_data, ratio=0.75)

# Print the final, combined rule set
print("--- Final Ruleset from the Expert Model ---")
expert_model.print_asp(simple=True)

# Get predictions from our new model
expert_predictions_tuples = expert_model.predict(X_test)
expert_predicted_labels = [p[0] for p in expert_predictions_tuples]

# Calculate accuracy
expert_correct_predictions = 0
for i in range(len(Y_test)):
    if expert_predicted_labels[i] == Y_test[i]:
        expert_correct_predictions += 1

expert_accuracy = expert_correct_predictions / len(Y_test)

print("--- Baseline Model Evaluation ---")
#print(f"True Labels:      {Y_test}")
#print(f"Predicted Labels: {predicted_labels}")
print(f"Accuracy: {accuracy * 100:.2f}%\n")


print("--- Expert Model Evaluation ---")
#print(f"True Labels:      {Y_test}")
#print(f"Predicted Labels: {expert_predicted_labels}")
print(f"Accuracy: {expert_accuracy * 100:.2f}%")

# Instantiate a new classifier
learned_confidence_model = Classifier(attrs=model_template.attrs, numeric=model_template.numeric, label=model_template.label)

# Define our expert rules as strings, but WITHOUT the 'with confidence' part.
#rule1_no_confidence = "class = 'Higher_risk' if 'Range_size' '<=' '5'"
#rule2_no_confidence = "class = 'Higher_risk' if 'Clutch_size' '<=' '1'"

# Add the manual rules to the model
#learned_confidence_model.add_manual_rule(rule1_no_confidence, model_template.attrs, model_template.numeric, ['Lower_risk', 'Higher_risk'], instructions=False)
#learned_confidence_model.add_manual_rule(rule2_no_confidence, model_template.attrs, model_template.numeric, ['Lower_risk', 'Higher_risk'], instructions=False)

print("--- Manual Rules Added (Before Training) ---")
print("Notice the default confidence value of 0.5 assigned to each rule.")
#for rule in learned_confidence_model.rules:
#    print(rule)

# Now, fit the model on the training data.
# The algorithm will calculate the confidence of our provided rules and then learn any additional rules needed.
learned_confidence_model.fit(train_data, ratio=0.5)

# Print the final, combined rule set
print("--- Final Ruleset with Learned Confidence ---")
print("The confidence values have now been updated based on the training data!")
learned_confidence_model.print_asp(simple=True)
            #Note that confidence values will be relatively low due to the small size of the training data. 

# Get predictions from our new model
learned_conf_predictions = learned_confidence_model.predict(X_test)
learned_conf_labels = [p[0] for p in learned_conf_predictions]

# Calculate accuracy
learned_conf_accuracy = sum(1 for i in range(len(Y_test)) if learned_conf_labels[i] == Y_test[i]) / len(Y_test)

print("--- Learned Confidence Model Evaluation ---")
print(f"True Labels:      {Y_test}")
print(f"Predicted Labels: {learned_conf_labels}")
print(f"Accuracy: {learned_conf_accuracy * 100:.2f}%")

# First, let's re-print the rules from our expert model for comparison
print("--- Rules Before Pruning ---")
expert_model.print_asp(simple=True)

############PRUNNING##################

# Method 1: Simple Post-Hoc Confidence Pruning: removing those rules with a low confidence according to me
# Import the prune_rules function from the core algorithm file
from algo import prune_rules

# Apply the pruning function
# This will create a new list containing only the rules that meet the confidence threshold.
pruned_rules = prune_rules(expert_model.rules, confidence=0.90)

# We can create a new model instance to hold these pruned rules
simple_pruned_model = Classifier(attrs=model_template.attrs, numeric=model_template.numeric, label=model_template.label)
simple_pruned_model.rules = pruned_rules

print("\n--- Rules After Pruning (Confidence >= 0.90) ---")
simple_pruned_model.print_asp(simple=True)
            
# Instantiate a new model for this experiment
advanced_pruning_model = Classifier(attrs=model_template.attrs, numeric=model_template.numeric, label=model_template.label)

##################
#### Method 2: Advanced Confidence-Driven Learning

# Now, train using confidence_fit with a high 15% improvement threshold
print("--- Training with confidence_fit(improvement_threshold=0.15) ---")
advanced_pruning_model.confidence_fit(train_data, improvement_threshold=0.15)



--- Final Ruleset from the Expert Model ---
extinction_risk(X,'lower_risk') :- range_size(X,N34), N34>54913.11287, not ab2(X), not ab9(X), not ab10(X), not ab14(X), not ab18(X), not ab20(X), not ab25(X). [confidence: 0.94924]
extinction_risk(X,'lower_risk') :- range_size(X,N34), N34<=54825.57, not ab28(X), not ab29(X), not ab35(X), not ab52(X), not ab55(X), not ab57(X), not ab60(X), not ab64(X), not ab68(X), not ab69(X), not ab70(X), not ab71(X). [confidence: 0.81371]
extinction_risk(X,'higher_risk') :- range_size(X,N34), N34>296.85. [confidence: 0.9878]
extinction_risk(X,'higher_risk') :- range_size(X,N34), N34<=97.93, not ab76(X), not ab77(X), not ab79(X), not ab80(X). [confidence: 0.96786]
extinction_risk(X,'higher_risk') :- range_size(X,N34), N34>98.0. [confidence: 0.64173]
extinction_risk(X,'lower_risk') :- not primary.lifestyle(X,'aerial'). [confidence: 0.85]
extinction_risk(X,'lower_risk') :- primary.lifestyle(X,'aerial'). [confidence: 0.55]
ab1(X) :- hand.wing.index(X,N23), N23

In [9]:
print("\n--- Rules Learned via Confidence-Driven Learning ---")
print("Note how the model is simpler and did not learn any exceptions to rules or `abnormalities', as they did not meet the high confidence improvement threshold.")
advanced_pruning_model.print_asp(simple=True)


--- Rules Learned via Confidence-Driven Learning ---
Note how the model is simpler and did not learn any exceptions to rules or `abnormalities', as they did not meet the high confidence improvement threshold.
extinction_risk(X,'lower_risk') :- range_size(X,N34), N34>54913.11287. [confidence: 0.93064]
extinction_risk(X,'lower_risk') :- range_size(X,N34), N34>327.84. [confidence: 0.73745]
extinction_risk(X,'higher_risk') :- kipps.distance(X,N21), N21>15.4. [confidence: 0.84688]
extinction_risk(X,'higher_risk') :- not rlm(X,'a'), range_size(X,N34), N34<=23.1. [confidence: 0.78125]
extinction_risk(X,'lower_risk') :- rlm(X,'a'). [confidence: 0.77083]
extinction_risk(X,'higher_risk') :- elevational.range(X,N29), N29<=1050.0, body_mass(X,N35), N35>29.8. [confidence: 0.68519]
extinction_risk(X,'higher_risk') :- beak.length_nares(X,N16), N16<=6.8. [confidence: 0.67857]
extinction_risk(X,'lower_risk') :- beak.width(X,N17), N17>2.8, elevational.range(X,N29), N29>44.0, elevational.range(X,N29), N

In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [11]:
clf = DecisionTreeClassifier(max_depth=4, random_state=0)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

cm = confusion_matrix(y_test, y_pred)

ValueError: could not convert string to float: 'Terrestrial'